# Day 27 — DENSE Reproduction Discrepancy Analysis

**Week 4: Paper → Repo → Baseline → Experiment**

## Today's question

Yesterday we proved that the released DENSE repository can be pushed through an end-to-end smoke path:

$$
\text{Cora}
\rightarrow
\text{Bundle Sampling}
\rightarrow
\text{LLM Query}
\rightarrow
\text{Bundle Supervision}
\rightarrow
\text{GNN Training}
\rightarrow
\text{Evaluation}
$$

The smoke run produced:

- Dataset: **Cora**
- GNN: **GCN**
- Query model: **DeepSeek-V4.1-Flash**
- Bundle size: **5**
- Number of bundles: **5**
- Loss: **average**
- Stages: **10**
- Test accuracy: **0.4022**

This is **not** an official reproduction.

Today we do not chase a higher number blindly. We answer:

> **Why is our smoke run different from the paper/README setup, and which discrepancy should be resolved first?**

## 1. Research objective

By the end of Day 27, I should be able to distinguish:

1. **Expected experimental differences** — caused by intentionally using a tiny smoke configuration.
2. **Provider/model differences** — caused by replacing the paper's LLM.
3. **Repository inconsistencies** — README/paper/code paths that do not agree.
4. **Our compatibility fixes** — modifications needed only to make the released code execute.
5. **Algorithmic discrepancies** — differences that could materially change DENSE itself.

### Rule for today

Do **not** modify several things at once.

First build an evidence-backed discrepancy table. Then select **one** discrepancy for the next experiment.

## 2. Freeze the Day 26 baseline

Record the exact successful smoke configuration before touching anything.

```text
python bundle.py   --device 0   --dataset cora   --bundle_size 5   --num_samples 5   --sample_criterion neighbor   --max_hop 2   --query_type gpt   --model deepseek/deepseek-v4.1-flash   --loss_type average   --gnn_type gcn   --stages 10   --lr 0.001   --wd 0.001   --repeat 1
```

### Baseline anchor

| Item | Day 26 smoke run |
|---|---|
| Dataset | Cora |
| Bundle size | 5 |
| Number of bundles | 5 |
| Sampling | neighbor |
| Max hop | 2 |
| Query model | DeepSeek-V4.1-Flash |
| Loss | average |
| GNN | GCN |
| Stages | 10 |
| Resampling/refinement | No |
| Accuracy | 0.4022 |

### Q1
Why must we freeze this configuration before debugging discrepancies?

> Because this configuration has been verified to run successfully, it serves as a fixed baseline. By changing one factor at a time, we can isolate the cause of each discrepancy and make the comparison reproducible.

## 3. Build the three-way specification map

A reproduction can have **three different sources of truth**:

```text
Paper
  ↓
README / official commands
  ↓
Released implementation
```

They are not automatically identical.

For every important setting, compare all three instead of assuming the README is executable truth.

### Working table

| Component | Paper | README / official Cora command | Released code | Day 26 run | Status |
|---|---|---|---|---|---|
| Dataset | Cora | Cora | supported | Cora | aligned |
| Bundle size | inspect | 5 | inspect | 5 | TBD |
| Num. bundles | inspect | 100 | supported | 5 | intentional smoke difference |
| Sampling | inspect | neighbor | supported | neighbor | likely aligned |
| Max hop | inspect | 2 | supported | 2 | likely aligned |
| Query model | GPT-4o-era setup | GPT-4o | OpenAI-compatible query helper | DeepSeek-V4.1-Flash | provider/model difference |
| Loss | inspect | ranking | inspect implementation | average | intentional + needs investigation |
| GNN | inspect | GIN | model factory must be checked | GCN | **potential repo discrepancy** |
| Stages | inspect | 400 100 100 | inspect control flow | 10 | intentional smoke difference |
| Refinement | DENSE component | `--resample` | inspect path | disabled | intentional smoke difference |
| Evaluation | inspect | implicit | inspect code | test accuracy | TBD |

### Q2
Which rows are merely expected smoke-test differences, and which rows suggest that the released repository itself may be inconsistent?

> Expected smoke-test differences include the number of bundles, loss type, training stages, and the disabled refinement/resampling stage. The query model is also an experimental provider/model difference because we used DeepSeek instead of GPT-4o.

> Potential repository inconsistencies include the GNN backbone, because the README specifies GIN while the released model factory may not implement it, and the refinement/resampling path, whose implementation still needs to be verified.

## 4. Inspect discrepancy A — README says GIN, code-supported backbones

This is our first high-priority repository question.

Do not change the model yet. First collect evidence.

Run in the **original repository checkout**:

```bash
grep -n -A 35 "def prepare_model" utils.py
grep -Rn --exclude-dir=.git "gnn_type\|GIN\|gin" README.md bundle.py utils.py model/
```


### Q3
Does the released `prepare_model()` actually implement the backbone requested by the official Cora command?

> No. The released prepare_model() only implements gcn and glognn, while the official Cora command requests gin. Therefore, the released model factory cannot construct the backbone specified by the README.

### Q4
If README requests GIN but the model factory cannot construct GIN, what kind of discrepancy is this?

> This is a README–implementation inconsistency (or documentation–implementation discrepancy). The official reproduction command specifies a model configuration that is not supported by the released model factory.

## 5. Inspect discrepancy B — `average` vs `ranking` loss

Day 26 deliberately used `average` because it was the smallest understandable bundle-level CE path.

Now inspect the released loss implementation:

```bash
grep -n -A 45 "def bundle_loss" bundle.py
```

### Paper-level expectation

From our Day 24 reading:

- $L_{BE}$ encourages the bundle-level prediction to match the LLM-provided bundle label.
- $L_R$ adds a ranking constraint so the LLM bundle label should become top-ranked.
- The official reproduction path should be compared carefully with the released `ranking` branch.

### Code evidence

```text

```

### Q5
What exactly does `loss_type=average` compute in the released code?

> loss_type=average first averages the node logits within each bundle and then applies cross-entropy between the averaged bundle logits and the LLM-provided bundle label. It therefore implements bundle-level supervision rather than assigning the bundle label independently to every node.

### Q6
What exactly does `loss_type=ranking` compute?

> loss_type=ranking computes two terms. First, it converts each node's logits into probabilities using softmax and averages these probabilities within each bundle. It then compares the probability of the LLM-provided bundle class with the maximum class probability. If the LLM-provided class is not top-ranked, a ranking penalty is added. Second, it computes cross-entropy between the averaged bundle logits and the LLM-provided bundle label. Therefore, the ranking loss combines bundle-level cross-entropy with a ranking penalty.

### Q7
Does the implementation match our paper-level understanding of $L_{BE}+L_R$?

> Yes, at the conceptual level. The released ranking branch combines a bundle-level cross-entropy term with a ranking penalty. The cross-entropy term encourages the averaged bundle prediction to match the LLM-provided label, corresponding to $L_{BE}$, while the ranking term penalizes cases where the LLM-provided class is not the top-ranked class, corresponding to $L_R$. Therefore, the implementation is consistent with our paper-level understanding of $L_{BE}+L_R$.

## 6. Inspect discrepancy C — stages and refinement/resampling

The smoke run intentionally used one short stage. DENSE's full method contains refinement.

Inspect the control flow:

```bash
sed -n '20,125p' bundle.py
grep -Rn --exclude-dir=.git "resample\|bundle_resample\|stages" bundle.py
```

### Draw the actual released execution path

Complete this from code evidence:

```text
solve()
  ↓
bundle_presample()
  ↓
batch_bundle_query()
  ↓
for each stage:
    bundle_optimize()
        ↓
    if not the last stage:
        bundle_resample()   ← CALLED BUT NOT DEFINED
  ↓
evaluation
```

### Q8
When does refinement/resampling occur?

> Refinement/resampling is intended to occur after each non-final training stage and before the next optimization stage. For --stages 400 100 100, it should occur after the 400-epoch stage and again after the first 100-epoch stage.

### Q9
Can the released multi-stage path execute as written, or is another missing/stale interface involved?

> No. The released multi-stage path cannot execute as written because solve() calls self.bundle_resample(...) between stages, but no bundle_resample() method is defined in the released upstream/master. This indicates an incomplete or stale refinement interface in the released implementation.

## 7. Inspect discrepancy D — LLM provider/model

Our Day 26 run proved the query pipeline works with an OpenAI-compatible endpoint, but:

```text
GPT-4o ≠ DeepSeek-V4.1-Flash
```

This difference may change pseudo-label quality and therefore downstream GNN accuracy.

### Important distinction

This is **not automatically a repository bug**.

It is an experimental-condition difference.

### Q10
Why can changing only the query LLM change the final GNN result even if the GNN code is identical?

> Because the query LLM generates the bundle-level pseudo-labels used to supervise the GNN. Different LLMs may assign different labels or have different pseudo-label quality, which changes the training loss and gradients even when the GNN architecture and training code are identical. Therefore, the final GNN accuracy can also change.

### Q11
What should we hold constant if we later compare two query LLMs fairly?

> To compare two query LLMs fairly, we should keep the dataset, sampled bundles, prompts, sampling strategy, GNN architecture, loss function, training stages, hyperparameters, random seeds, and evaluation protocol fixed. Ideally, only the query LLM should change.

## 8. Classify every discrepancy

Use these labels:

- **I — Intentional smoke simplification**
- **P — Provider/environment difference**
- **R — Released-repository inconsistency**
- **C — Compatibility fix introduced by us**
- **A — Possible algorithmic mismatch requiring paper/code verification**

| Discrepancy | Label | Evidence | Could affect accuracy? | Must resolve before official reproduction? |
|---|---|---|---|---|
| 5 vs 100 bundles | **I** | Day 26 smoke uses 5 bundles, while the README Cora command uses 100 | Yes | Yes |
| DeepSeek vs GPT-4o | **P** | Day 26 uses DeepSeek-V4.1-Flash, while the official setup uses GPT-4o | Yes | Yes |
| average vs ranking | **I** | Day 26 deliberately uses `average`; README requests `ranking`; both branches exist in `bundle_loss()` | Yes | Yes |
| GCN vs README GIN | **I + R** | Day 26 deliberately uses GCN, but README requests GIN while released `prepare_model()` only supports GCN/GloGNN | Yes | Yes |
| 10 vs 400/100/100 stages | **I** | Day 26 uses a single 10-epoch stage, while README specifies `400 100 100` | Yes | Yes |
| refinement disabled | **I + R** | Day 26 intentionally bypasses refinement; released `solve()` calls `bundle_resample()`, but no `bundle_resample()` definition exists in `upstream/master` | Yes | Yes |
| query helper signature fix | **C + R** | A released query call/interface inconsistency had to be repaired for the smoke pipeline | Possibly indirect | Document and verify |
| `bundle_optimize` signature fix | **C + R** | The released call site and function signature were inconsistent; our patch aligned them for smoke execution | Possibly | Document and verify |
| OpenAI-compatible endpoint / serial querying | **C + P** | A third-party OpenAI-compatible endpoint was used, and query concurrency was reduced to `max_workers=1` because of RPM limits | Usually not directly, but failed/missing queries may affect results | Document |
| Paper formula vs released `ranking` implementation | **A** | The code structurally contains bundle-level CE plus a ranking penalty, but exact equation-level equivalence with the paper has not yet been verified | Potentially | Verify |

### Current interpretation

The differences between the Day 26 smoke run and the official reproduction setup come from several distinct sources:

```text
Day 26 smoke run ≠ Official reproduction
                  |
        -------------------------
        |           |           |
        I           P           R
   intentional   provider /   released
   simplification environment repository
        |           |        inconsistency
        |           |           |
   5 bundles     DeepSeek    GIN missing
   average                   resample missing
   10 stages                 stale interfaces
   no refinement
                  |
                  A
          paper/code details
          still to verify

## 9. Rank discrepancies by scientific priority

Do not rank by “easiest to fix.” Rank by how strongly each one blocks a defensible reproduction.

### Priority rubric

$$
	ext{Priority}
pprox
	ext{Impact on method}
	imes
	ext{Impact on result}
	imes
	ext{Uncertainty}
$$

### My ranking

1. > **Missing refinement / `bundle_resample()` path** — refinement is a core component of DENSE, but the released multi-stage path cannot execute as written.
2. > **README GIN vs released model factory** — the official Cora backbone cannot be constructed by the released `prepare_model()`.
3. > **`average` vs `ranking` objective** — the smoke run uses a simplified loss instead of the official ranking objective.
4. > **GPT-4o vs DeepSeek-V4.1-Flash** — changing the LLM changes the pseudo-supervision provided to the GNN.
5. > **Smoke-scale vs official-scale configuration** — 5 vs 100 bundles and 10 vs 400/100/100 stages can substantially change training and evaluation results.

### Q12
Which **single discrepancy** should be investigated experimentally first, and why?

> **The `average` vs `ranking` loss discrepancy should be investigated experimentally first.** Although the missing refinement path has higher scientific priority, the ranking loss is already implemented in the released repository and can be changed while holding almost every other factor constant. This gives us a clean single-variable experiment to measure how the official objective changes the smoke baseline before making larger implementation repairs.

## 10. One-variable experiment plan

Do **not** run the full official experiment yet.

The first controlled experiment changes only the loss function from the Day 26 smoke baseline.

### Hypothesis

> Using the released `ranking` loss may change the final GNN accuracy because it adds a ranking constraint in addition to bundle-level cross-entropy. This more closely matches the official README configuration than the `average` smoke loss.

### Independent variable

> Loss type: `average` → `ranking`.

### Controlled variables

> Keep the dataset, bundle size, number of bundles, sampling criterion, max hop, query LLM, GNN backbone, number of stages, learning rate, weight decay, and repeat count identical to the Day 26 smoke baseline.

### Metric

> Test accuracy, plus bundle valid rate / bundle class accuracy when relevant.

### Expected interpretation

If the result improves:

> The ranking objective may provide a more effective training signal than the simplified `average` objective under the current smoke configuration. This would justify keeping `ranking` as we move toward the official configuration.

If the result degrades:

> The ranking objective does not necessarily improve performance under the small smoke configuration. The result should not be interpreted as evidence against the paper because other experimental conditions still differ substantially from the official setup.

If the run crashes:

> The failure itself becomes repository/reproduction evidence; record the first real traceback before editing code.

### Experiment 1 result

**Status:** Failed before training completed.

The released `ranking` branch raises:

`TypeError: cross_entropy() missing 1 required positional argument: 'target'`

The ranking implementation calls:

`F.cross_entropy(torch.mean(bundle_logits, dim=1))`

without providing the target `bundle_classes`.

This is a newly confirmed released-repository inconsistency rather than an experimental accuracy result. The implementation should be checked against the paper's exact $L_{BE}+L_R$ formulation before applying a patch.

## 11. Reproduction integrity check

Before any new modification:

```bash
git status --short
git diff -- bundle.py queryhelper.py utils.py
git rev-parse HEAD
```

Keep two concepts separate:

```text
ORIGINAL RELEASED CODE
        ↓
document discrepancy
        ↓
MINIMAL COMPATIBILITY PATCH
        ↓
run experiment
```

Never silently turn patched code into “the official implementation.”

### Q13
Why is preserving the original commit + diff scientifically important?

> Preserving the original commit and the exact diff is scientifically important because it separates the released implementation from our own reproduction and compatibility fixes. The original commit provides a fixed reference for what the authors actually released, while the diff records exactly what we changed. This makes the experiment auditable and reproducible, and prevents us from accidentally attributing behavior introduced by our patches to the official DENSE implementation.

## 12. Day 27 result log

### Repository commit

> `<paste the output of git rev-parse HEAD here>`

> Upstream reference: `80834ed`  
> Reproduction branch: `fix/reproduction`

### Files inspected

> `README.md`, `bundle.py`, `queryhelper.py`, `utils.py`, and relevant `model/` files.

### Most important discrepancy found

> The released repository is not fully consistent with the official reproduction configuration and paper-level execution path. In particular, the README requests model/loss/refinement settings that are not completely executable as released. During Day 27, we identified issues in the released model factory, ranking-loss implementation, and multi-stage refinement path.

### Evidence

```text
1. README / model discrepancy
   - The official Cora command requests:
       --gnn_type gin
   - The released prepare_model() implementation only explicitly
     constructs GCN and GloGNN.
   - Therefore, the official README Cora command cannot construct
     the requested GIN backbone through the released model factory.

2. Ranking-loss discrepancy
   - The official command uses:
       --loss_type ranking
   - The released ranking branch reached a runtime error because
     F.cross_entropy(...) was called without the required target.
   - After a minimal correction, the ranking path became executable.

3. Refinement / multi-stage discrepancy
   - solve() calls bundle_resample(...) between training stages.
   - The released repository references bundle_resample(), but the
     corresponding implementation is missing/stale in the released code.
   - A minimal reproduction implementation allowed the multi-stage
     training/refinement path to execute.

4. Provider / scaling limitation
   - Small bundle-query runs completed successfully.
   - Scaling from 5 to 100 requested bundles with the current
     OpenAI-compatible DeepSeek endpoint produced APITimeoutError
     and repeated HTTP 429 rate_limit_rpm_exceeded errors.
   - This is classified as a provider/infrastructure limitation,
     not automatically as a DENSE algorithmic failure.
```

### Was code changed today?

> Yes. Minimal reproduction patches were introduced to investigate previously identified execution-path discrepancies, including the ranking-loss path and the multi-stage refinement path. An additional retry/backoff modification to queryhelper.py was also tested for external API instability, but this API-handling change remains experimental and should not yet be treated as part of the validated reproduction patch.

### Experiment run today?

> Yes. We progressively tested the reproduction pipeline rather than immediately running the full official configuration:

5 bundles with ranking loss;
5 bundles with multi-stage refinement;
5 bundles with 300 100 100 stages;
attempted scaling to 100 bundles;
API retry/backoff behavior was also tested after provider failures appeared.

### Result

> The corrected ranking path successfully executed on the 5-bundle smoke configuration and produced approximately 0.3911 test accuracy.

After enabling the repaired multi-stage refinement path and using 300 100 100 stages, the 5-bundle experiment reached approximately 0.4742 test accuracy.

The refinement path visibly reduced bundle sizes across stages:

[5, 5, 5, 5, 5]
     ↓
[4, 4, 4, 4, 4]
     ↓
[3, 3, 3, 3, 3]

However, the attempted 100-bundle run could not yet provide a clean official-scale result because the external LLM provider produced timeout and RPM rate-limit errors.

### Interpretation

> Day 27 showed that the gap between the Day 26 smoke baseline and a defensible official reproduction is caused by multiple independent factors rather than a single accuracy issue.

First, several released-repository inconsistencies affect executability, including the GIN model-factory mismatch, the ranking-loss runtime issue, and the missing/stale refinement path. Second, our experimental configuration still differs from the official setup, especially in the query LLM. Third, scaling the bundle-query stage exposed an external API-provider bottleneck.

Therefore, the current results should be treated as evidence that the patched DENSE pipeline can execute end-to-end, not as a reproduction of the paper's reported Cora result. A defensible official reproduction still requires resolving the official backbone/configuration question and obtaining a sufficiently stable LLM query environment.

## 13. Researcher's reflection

Day 27 showed that successfully executing code is not equivalent to reproducing a paper. A defensible reproduction requires alignment between the paper, released implementation, experimental configuration, and evaluation protocol.

The discrepancies observed today fall into two categories:

- **Software/repository issues:** unsupported README backbone, ranking-loss runtime inconsistency, and missing/stale refinement execution path.
- **Experimental/infrastructure differences:** query LLM, number of bundles, training stages, and API-provider stability.

Reading the paper first allowed us to recognize when executable code did not actually correspond to the intended DENSE method. Therefore, the current experiments demonstrate that our patched pipeline is executable, but they are not yet sufficient to claim an official Cora reproduction.

A reproduction claim will require a stable official-scale run with the intended configuration, documented code patches, and a result that can be meaningfully compared with the paper.

## Day 27 Completion Checklist

- [x] Frozen the Day 26 smoke baseline
- [x] Compared paper vs README vs released implementation
- [x] Verified the GNN-backbone discrepancy
- [x] Inspected `average` and `ranking` loss paths
- [x] Inspected multi-stage/refinement control flow
- [x] Classified discrepancies by source
- [x] Ranked discrepancies by scientific priority
- [x] Designed and executed a controlled `average → ranking` experiment
- [x] Preserved original commit and local diff
- [x] Wrote a defensible Day 27 conclusion

---

### Day 27 Final Answer

The Day 26 accuracy of `0.4022` is not directly comparable with the official DENSE result because the smoke run intentionally changed several experimental conditions, including the number of bundles, loss configuration, GNN backbone, training stages, refinement, and query LLM.

Day 27 further identified several released-repository inconsistencies, including the README/model-factory mismatch for GIN, a runtime issue in the released ranking-loss path, and a missing/stale multi-stage refinement path.

After minimal reproduction fixes, the ranking configuration and multi-stage refinement pipeline became executable. A small-scale `300 → 100 → 100` run reached `0.4742`, but this is still not an official reproduction because only 5 bundles were used and the query LLM/provider differs from the official setup.

Scaling toward 100 bundles exposed an additional infrastructure limitation: the current external LLM API produced timeouts and RPM rate-limit errors.

Therefore, the next controlled experiment should scale the number of bundles toward the official setting while keeping the validated ranking + GCN + multi-stage configuration fixed, once a sufficiently stable LLM API environment is available.

$$
\boxed{
\text{One discrepancy}
\rightarrow
\text{One controlled experiment}
\rightarrow
\text{One interpretable result}
}
$$